# Tome 1, Chapter 11: Ideals, a First Look at Algebraic Number Theory (Python companion)

Exercises first, solutions at the end. Computer-science threads: **exact arithmetic** in $\mathcal O_K$ with integer pairs,
**ideals as integer lattices** in Hermite normal form (the same data structure as in lattice cryptography),
**class numbers** by enumerating reduced quadratic forms, and **statistics** of how primes split.

*Pause and think before each solution: try to predict the output first.*

In [ ]:
import math
from math import gcd
from collections import Counter
import matplotlib.pyplot as plt
from sympy import primerange, isprime, factorint
import mpmath

## Exercises

**P11.1 (★) Arithmetic in $\mathcal O_K$.** Represent $m+n\omega_d$ as the pair $(m,n)$, with $\omega_d^2=t\omega_d+s$ (Proposition 11.2). Write `mul`, `conj`, `norm`, `trace`. Then check Proposition 11.2 by brute force: among all $\frac{u+v\sqrt d}{4}$ with $|u|,|v|\le 12$, the ones with integer trace and norm are exactly the elements of $\Z[\omega_d]$.

In [ ]:
# Your code here


**P11.2 (★★) Fundamental units.** For every squarefree $2\le d\le 100$, find the fundamental unit $\varepsilon_d$ of $\Q(\sqrt d)$ by searching for the smallest $v\ge1$ such that $\frac{u+v\sqrt d}2$ is a unit. Plot $\log\varepsilon_d$ against $d$. Which $d$ give a huge unit? *Guess first: is $\log\varepsilon_d$ more like $\log d$ or like $\sqrt d$?*

In [ ]:
# Your code here


**P11.3 (★★) Ideals as lattices.** By Lemma 11.5, an ideal is a lattice $\Z\alpha+\Z\beta$; its Hermite normal form is the unique basis $(a,0),(b,c)$ with $a,c>0$ and $0\le b<a$. Write `ideal(gens)`, `imul(I, J)` and `inorm(I)` (with Lemma 11.7). Check Proposition 11.6 ($P^2=(2)$, $Q\bar Q=(3)$, $PQ=(1+\sqrt{-5})$) and test the remark of Section 11.3: is $N(I)$ always the index $ac$?

In [ ]:
# Your code here


**P11.4 (★★) Class numbers.** For a negative discriminant $D$, Gauss's *reduced forms* are the $(a,b,c)$ with $b^2-4ac=D$, $|b|\le a\le c$, $b\ge0$ when $|b|=a$ or $a=c$, and $\gcd(a,b,c)=1$. Their number is the class number $h(D)$ (a correspondence proved in Tome 12; here we use it as a computing tool and check it against $h(-20)=2$, Theorem 11.17). List the imaginary quadratic fields $\Q(\sqrt d)$ with $h=1$ for $-3000\le d<0$, and compute $e^{\pi\sqrt{163}}$ to 35 digits.

In [ ]:
# Your code here


**P11.5 (★) How primes split.** For $d=-5$, classify every prime $p<10^5$ as ramified, inert, split with principal factors, or split with non-principal factors. Check Exercise 11.8 ($p=x^2+5y^2\iff p\equiv1,9\pmod{20}$) on all of them, and compute the proportion of each type. *Guess the proportions before running the code.*

In [ ]:
# Your code here


---
## Solutions

### P11.1
With $\omega^2=t\omega+s$: $(a+b\omega)(c+e\omega)=(ac+sbe)+(ae+bc+tbe)\omega$. Conjugation fixes $a$ and sends $\omega$ to $t-\omega$ (since $\omega+\bar\omega=t$).

In [ ]:
def ring(d):
    """Return (t, s) with omega^2 = t*omega + s for the ring of integers of Q(sqrt d)."""
    return (1, (d - 1) // 4) if d % 4 == 1 else (0, d)

def mul(x, y, d):
    t, s = ring(d)
    (a, b), (c, e) = x, y
    return (a * c + s * b * e, a * e + b * c + t * b * e)

def conj(x, d):
    t, _ = ring(d)
    a, b = x
    return (a + t * b, -b)

def norm(x, d):
    return mul(x, conj(x, d), d)[0]

def trace(x, d):
    t, _ = ring(d)
    return 2 * x[0] + t * x[1]

from fractions import Fraction as F
for d in [-5, -3, -1, 2, 5, 13, 10]:
    good = set()
    for u in range(-12, 13):
        for v in range(-12, 13):
            a, b = F(u, 4), F(v, 4)                       # the number a + b*sqrt(d)
            tr, nm = 2 * a, a * a - d * b * b
            if tr.denominator == 1 and nm.denominator == 1:
                good.add((a, b))
    # elements m + n*omega written as a + b*sqrt(d)
    t, _ = ring(d)
    expected = set()
    for (a, b) in good:
        n = 2 * b if t == 1 else b
        m = a - n / 2 if t == 1 else a
        assert m.denominator == 1 and n.denominator == 1, (d, a, b)
        expected.add((a, b))
    print(f"d = {d:3d}: {len(good)} numbers with integer trace and norm, all in Z[omega]")

### P11.2
A unit $\frac{u+v\sqrt d}2$ has $u^2-dv^2=\pm4$; when $d\not\equiv1\pmod 4$ we also need $u,v$ even. The least $v\ge1$ gives the fundamental unit, because the unit grows with $v$ (proof of Theorem 11.4).

In [ ]:
def is_squarefree(n):
    return all(e == 1 for e in factorint(n).values())

def fundamental_unit(d):
    v = 1
    while True:
        for sign in (-4, 4):
            u2 = d * v * v + sign
            if u2 > 0:
                u = math.isqrt(u2)
                if u * u == u2 and (d % 4 == 1 or (u % 2 == 0 and v % 2 == 0)):
                    return u, v          # epsilon = (u + v sqrt d) / 2
        v += 1

units = {d: fundamental_unit(d) for d in range(2, 101) if is_squarefree(d)}
logs = {d: math.log((u + v * math.sqrt(d)) / 2) for d, (u, v) in units.items()}
for d in [2, 3, 5, 13, 46, 61, 94]:
    u, v = units[d]
    print(f"d = {d:3d}: epsilon = ({u} + {v} sqrt {d})/2,  log epsilon = {logs[d]:.2f}")
plt.figure(figsize=(8, 3.5))
plt.scatter(list(logs), list(logs.values()), s=12)
xs = range(2, 101)
plt.plot(xs, [math.sqrt(x) * math.log(x) for x in xs], 'r--', lw=1, label=r'$\sqrt{d}\,\log d$ (for scale)')
plt.xlabel('d'); plt.ylabel(r'$\log\varepsilon_d$'); plt.legend(); plt.title('Fundamental units of real quadratic fields')
plt.show()

The size of $\log\varepsilon_d$ jumps around wildly: tiny for $d=m^2\pm1$ or $m^2\pm4$, enormous for $d=46$ or $94$. Note $d=61$: Chapter 8 found the huge Pell solution $1766319049+226153980\sqrt{61}$, but in $\mathcal O_K=\Z\left[\frac{1+\sqrt{61}}2\right]$ the fundamental unit is the small $\frac{39+5\sqrt{61}}2$, and the Pell solution is its cube (check it!). The dashed curve $\sqrt d\log d$ is the right order of magnitude for the largest units: the class number formula of Tome 12, $h_d\log\varepsilon_d=\frac{\sqrt D}2L(1,\chi_D)$ with $L(1,\chi_D)\ll\log D$, gives $\log\varepsilon_d\ll\sqrt d\log d$, and shows that a huge unit forces a small class number.

### P11.3
The lattice of $(\gamma_1,\dots,\gamma_k)$ is spanned over $\Z$ by the $\gamma_i$ and $\gamma_i\omega$. To put it in Hermite normal form, run Euclid on the second coordinates, then take the gcd of what is left on the first axis.

In [ ]:
def hnf(vecs):
    pivot, rest = None, []
    for v in vecs:
        if v[1] == 0:
            rest.append(v); continue
        if pivot is None:
            pivot = v; continue
        while v[1] != 0:                                  # Euclid on second coordinates
            q = pivot[1] // v[1]
            pivot, v = v, (pivot[0] - q * v[0], pivot[1] - q * v[1])
        rest.append(v)
    a = 0
    for v in rest:
        a = gcd(a, v[0])
    if pivot[1] < 0:
        pivot = (-pivot[0], -pivot[1])
    return (a, pivot[0] % a, pivot[1])

def ideal(gens, d):
    vecs = []
    for g in gens:
        vecs += [g, mul(g, (0, 1), d)]
    return hnf(vecs)

def basis(I):
    a, b, c = I
    return [(a, 0), (b, c)]

def imul(I, J, d):
    return hnf([mul(x, y, d) for x in basis(I) for y in basis(J)])

def inorm(I, d):                                         # Lemma 11.7
    al, be = basis(I)
    return gcd(gcd(norm(al, d), norm(be, d)), trace(mul(al, conj(be, d), d), d))

d = -5
P, Q, Qb = ideal([(2, 0), (1, 1)], d), ideal([(3, 0), (1, 1)], d), ideal([(3, 0), (1, -1)], d)
print("P^2 = (2):", imul(P, P, d) == ideal([(2, 0)], d))
print("Q Qbar = (3):", imul(Q, Qb, d) == ideal([(3, 0)], d))
print("P Q = (1 + sqrt -5):", imul(P, Q, d) == ideal([(1, 1)], d))
print("HNF of P:", P, " norm", inorm(P, d), " index", P[0] * P[2])

import random
random.seed(0)
bad = 0
for _ in range(3000):
    d = random.choice([-1, -2, -3, -5, -6, -7, -14, -23, 2, 3, 5, 10, 13, 79])
    g1 = (random.randint(-30, 30), random.randint(-30, 30))
    g2 = (random.randint(-30, 30), random.randint(-30, 30))
    if g1 == (0, 0) and g2 == (0, 0):
        continue
    I = ideal([g1, g2], d)
    bad += inorm(I, d) != I[0] * I[2]
print("ideals where norm != index:", bad)

The norm of Lemma 11.7 always equals the index $ac=[\mathcal O:I]$, as the remark of Section 11.3 claims (proved in Tome 12). The Hermite normal form is also how lattice-based cryptography stores lattices: it is a canonical form, so two ideals are equal exactly when their HNFs are.

### P11.4

In [ ]:
def class_number(D):
    h, a = 0, 1
    while 3 * a * a <= -D:                         # reduced forms have a <= sqrt(|D|/3)
        for b in range(-a + 1, a + 1):
            if (b * b - D) % (4 * a) == 0:
                c = (b * b - D) // (4 * a)
                if c >= a and gcd(gcd(a, abs(b)), c) == 1 and not (a == c and b < 0):
                    h += 1
        a += 1
    return h

def fundamental_disc(d):
    return d if d % 4 == 1 else 4 * d

for d in [-1, -2, -3, -5, -6, -14, -23, -47, -71]:
    print(f"d = {d:4d}: h = {class_number(fundamental_disc(d))}")

h1 = [d for d in range(-1, -3001, -1) if is_squarefree(-d) and class_number(fundamental_disc(d)) == 1]
print("class number 1:", h1)
mpmath.mp.dps = 40
print("exp(pi sqrt 163) =", mpmath.exp(mpmath.pi * mpmath.sqrt(163)))

We find exactly the nine fields of Heegner, Baker and Stark; their theorem says that no other $d$, however large, has class number $1$. The near-integer $e^{\pi\sqrt{163}}=262537412640768743.99999999999925\ldots$ comes from the $j$-function: since $h(-163)=1$, $j\left(\frac{1+\sqrt{-163}}2\right)$ is an integer, and $j(\tau)=e^{-2i\pi\tau}+744+O(e^{2i\pi\tau})$.

### P11.5
By Theorem 11.11, an odd $p\ne5$ splits iff $\left(\frac{-5}p\right)=1$. A split factor is principal iff $p=x^2+5y^2$ (Exercise 11.5); we test that directly.

In [ ]:
N = 10**5
squares = {x * x for x in range(math.isqrt(N) + 1)}
def is_x2_5y2(p):
    return any(p - 5 * y * y in squares for y in range(math.isqrt(p // 5) + 1))

kinds = Counter()
for p in primerange(3, N):
    if p == 5:
        kinds['ramified'] += 1; continue
    if pow(-5 % p, (p - 1) // 2, p) != 1:
        kinds['inert'] += 1; continue
    principal = is_x2_5y2(p)
    assert principal == (p % 20 in (1, 9))
    kinds['split, principal' if principal else 'split, non-principal'] += 1
kinds['ramified'] += 1                       # p = 2
total = sum(kinds.values())
for k, v in kinds.items():
    print(f"{k:22s} {v:6d}  {v / total:.3f}")
print("Exercise 11.8 holds for every prime below", N)

The proportions are about $\frac12$ inert, $\frac14$ principal split and $\frac14$ non-principal split. That is Dirichlet's theorem on primes in progressions modulo $20$ (Tome 5), and, in the language of Tome 12, Chebotarev's theorem for the *Hilbert class field* of $\Q(\sqrt{-5})$: a prime splits into principal ideals with density $\frac1{2h}=\frac14$.